In [8]:
from pymongo import MongoClient
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, round

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]

print("Koneksi ke MongoDB berhasil!")

Koneksi ke MongoDB berhasil!


In [3]:
koleksi_menu.delete_many({})

data_menu = [
    {"nama_menu": "Espresso", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 100},
    {"nama_menu": "Americano", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 150},
    {"nama_menu": "Kopi Tubruk", "harga": 15000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 120},
    {"nama_menu": "Latte", "harga": 28000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 130},
    {"nama_menu": "Cold Brew", "harga": 32000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 60},

    {"nama_menu": "Ayam Geprek", "harga": 12000, "kategori": "Makanan", "pedas_level": 3, "rekomendasi": True},
    {"nama_menu": "Nasi Goreng", "harga": 20000, "kategori": "Makanan", "pedas_level": 1, "rekomendasi": True},
    {"nama_menu": "Mie Ayam", "harga": 10000, "kategori": "Makanan", "pedas_level": 1, "rekomendasi": True},
    {"nama_menu": "Roti Bakar", "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "rekomendasi": False},
    {"nama_menu": "Cireng Pedas", "harga": 5000, "kategori": "Makanan", "pedas_level": 2, "rekomendasi": False},

    {"nama_menu": "Teh Tarik", "harga": 6000, "kategori": "Non_Kopi", "stok": 50},
    {"nama_menu": "Matcha Ice", "harga": 20000, "kategori": "Non_Kopi", "stok": 60},
]

koleksi_menu.insert_many(data_menu)
print(f"Berhasil menyimpan {koleksi_menu.count_documents({})} dokumen ke collection 'menu'.")

Berhasil menyimpan 12 dokumen ke collection 'menu'.


In [4]:
print("Menu Kopi dengan tingkat kepahitan > 3")
for item in koleksi_menu.find({"tingkat_kepahitan": {"$gte": 3}}):
    print(f"{item['nama_menu']} (Kepahitan: {item['tingkat_kepahitan']})")

Menu Kopi dengan tingkat kepahitan > 3
Espresso (Kepahitan: 5)
Americano (Kepahitan: 4)
Kopi Tubruk (Kepahitan: 5)
Cold Brew (Kepahitan: 4)


In [16]:
print("Menu dengan harga diantara 15000 dan 30000")
for item in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"{item['nama_menu']} (Rp: {item['harga']:,})")

Menu dengan harga diantara 15000 dan 30000
Espresso (Rp: 18,000)
Americano (Rp: 22,000)
Kopi Tubruk (Rp: 15,000)
Latte (Rp: 28,000)
Nasi Goreng (Rp: 20,000)
Roti Bakar (Rp: 15,000)
Matcha Ice (Rp: 20,000)


In [5]:
print("Menu yang tidak memiliki field 'pedas_level'")
for item in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"{item['nama_menu']} {item['kategori']}")

Menu yang tidak memiliki field 'pedas_level'
Espresso Kopi
Americano Kopi
Kopi Tubruk Kopi
Latte Kopi
Cold Brew Kopi
Teh Tarik Non_Kopi
Matcha Ice Non_Kopi


In [6]:
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})

koleksi_menu.update_one({"nama_menu": "Cold Brew"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Ayam Geprek"}, {"$set": {"status": "Habis"}})

print("Hasil pembaruan status menu")
for item in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"{item['nama_menu']}: {item['status']}") 

Hasil pembaruan status menu
Espresso: Tersedia
Americano: Tersedia
Kopi Tubruk: Tersedia
Latte: Tersedia
Cold Brew: Habis
Ayam Geprek: Habis
Nasi Goreng: Tersedia
Mie Ayam: Tersedia
Roti Bakar: Tersedia
Cireng Pedas: Tersedia
Teh Tarik: Tersedia
Matcha Ice: Tersedia


In [17]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, round

spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

dokumen = list(koleksi_menu.find({}))

df_pandas = pd.DataFrame(dokumen)

df_pandas["_id"] = df_pandas["_id"].astype(str)

for col in ["pedas_level", "tingkat_kepahitan", "stok"]:
    if col in df_pandas.columns:
        df_pandas[col] = df_pandas[col].fillna(0).astype(int)

if "rekomendasi" in df_pandas.columns:
    df_pandas["rekomendasi"] = df_pandas["rekomendasi"].fillna(False).astype(bool)

df_spark = spark.createDataFrame(df_pandas)

print("Schema Spark DataFrame:")
df_spark.printSchema()

print("\nRata-rata Harga per Kategori:")
df_spark.groupBy("kategori").agg(
    round(avg("harga"), 2).alias("rata_rata_harga")
).show()

spark.stop()

/tmp/ipykernel_6000/3234683294.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_pandas["rekomendasi"] = df_pandas["rekomendasi"].fillna(False).astype(bool)


Schema Spark DataFrame:
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: long (nullable = true)
 |-- stok: long (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: long (nullable = true)
 |-- rekomendasi: boolean (nullable = true)


Rata-rata Harga per Kategori:


+--------+---------------+
|kategori|rata_rata_harga|
+--------+---------------+
|    Kopi|        23000.0|
| Makanan|        12400.0|
|Non_Kopi|        13000.0|
+--------+---------------+



Tantangan utama saat mengonversi data dari MongoDB ke Spark DataFrame terletak pada perbedaan karakteristik skema kedua teknologi tersebut. MongoDB beroperasi secara fleksibel, di mana setiap dokumen dalam collection yang sama dapat memiliki struktur data dan field yang berbeda-beda. Sebaliknya, Spark DataFrame membutuhkan skema data yang terstruktur dan konsisten di seluruh barisnya.

Ketika dokumen MongoDB dikonversi melalui Pandas sebelum dibaca oleh Spark, field-field yang sifatnya opsional atau hanya dimiliki oleh kategori tertentu (seperti `tingkat_kepahitan` pada menu Kopi atau `pedas_level` pada menu Makanan) secara otomatis diisi dengan nilai `NaN` atau `None` (`null` dalam PySpark) untuk baris yang tidak memilikinya.

Cara mengatasinya adalah dengan memastikan bahwa proses transformasi melalui Pandas menangani tipe data dengan benar (misalnya mengonversi tipe `ObjectId` menjadi *string*). Dengan begitu, Spark dapat melakukan inferensi tipe data (*schema inference*) secara otomatis tanpa menimbulkan kesalahan *type mismatch* saat membuat DataFrame.